In [1]:
import sumolib

ruta_net = "../data/processed/usaquen.net.xml"

net = sumolib.net.readNet(ruta_net)
nodos = []
for node in net.getNodes():
    x, y = node.getCoord()
    lon, lat = net.convertXY2LonLat(x, y)
    nodos.append({"id": node.getID(), "lon": lon, "lat": lat})

print(len(nodos), "nodos cargados")

12989 nodos cargados


In [ ]:
import requests
import time

URL = "https://serviciosgis.catastrobogota.gov.co/arcgis/rest/services/topografia/modelodigitalterrenobogotaurbano/MapServer/identify"

Z_MIN_BOGOTA = 2400  # margen por debajo del mínimo real (~2548)
Z_MAX_BOGOTA = 3600  # margen por encima del máximo real (~3184)

def consultar_elevacion(lon, lat, reintentos=3):
    params = {
        "geometry": f"{lon},{lat}",
        "geometryType": "esriGeometryPoint",
        "sr": 4326,
        "layers": "all",
        "tolerance": 1,
        "mapExtent": f"{lon-0.001},{lat-0.001},{lon+0.001},{lat+0.001}",
        "imageDisplay": "400,400,96",
        "returnGeometry": "false",
        "f": "json"
    }
    for intento in range(reintentos):
        try:
            r = requests.get(URL, params=params, timeout=20)
            data = r.json()
            if data.get("results"):
                val = data["results"][0]["attributes"].get("Stretch.Pixel Value")
                if val is None or val == "NoData":
                    return None
                try:
                    zf = float(val)
                except ValueError:
                    return None
                if zf < Z_MIN_BOGOTA or zf > Z_MAX_BOGOTA:
                    return None  # valor fuera de rango físico (dato inválido)
                return zf
            return None
        except requests.exceptions.RequestException:
            time.sleep(2)
    return None

In [3]:
import time
t0 = time.time()
z = consultar_elevacion(nodos[0]["lon"], nodos[0]["lat"])
print(z, time.time() - t0, "segundos")

2549.013916 0.5398752689361572 segundos


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import json
import os

RUTA_CHECKPOINT = "../data/processed/elevaciones_nodos.json"

# Si ya existe un checkpoint de un intento anterior, cargarlo
if os.path.exists(RUTA_CHECKPOINT):
    with open(RUTA_CHECKPOINT, "r") as f:
        resultados = json.load(f)
    print("Checkpoint cargado:", len(resultados), "resultados previos")
else:
    resultados = {}

cache = {}

def clave(lon, lat):
    return (round(lon, 5), round(lat, 5))

def consultar_con_cache(n):
    k = clave(n["lon"], n["lat"])
    if k not in cache:
        try:
            cache[k] = consultar_elevacion(n["lon"], n["lat"])
        except Exception:
            cache[k] = None
    return n["id"], cache[k]

#Solo consulta los nodos que no están ya en el checkpoint
pendientes = [n for n in nodos if n["id"] not in resultados]
print("Pendientes por consultar:", len(pendientes))

contador = 0
with ThreadPoolExecutor(max_workers=10) as executor:
    futuros = {executor.submit(consultar_con_cache, n): n for n in pendientes}
    for f in tqdm(as_completed(futuros), total=len(pendientes)):
        nid, z = f.result()
        resultados[nid] = z
        contador += 1
        if contador % 500 == 0:
            with open(RUTA_CHECKPOINT, "w") as fp:
                json.dump(resultados, fp)

# Guardado final
with open(RUTA_CHECKPOINT, "w") as fp:
    json.dump(resultados, fp)

for n in nodos:
    n["z"] = resultados.get(n["id"])

con_z = [n for n in nodos if n.get("z") is not None]
print("Nodos con z:", len(con_z), "de", len(nodos))

Pendientes por consultar: 12989


100%|██████████| 12989/12989 [34:53<00:00,  6.20it/s] 


Nodos con z: 12907 de 12989


In [ ]:
import xml.etree.ElementTree as ET

Z_MIN_BOGOTA = 2400
Z_MAX_BOGOTA = 3600

tree = ET.parse("../data/processed/usaquen_plain_z.nod.xml")  #parte del que ya tiene Z
root = tree.getroot()

corregidos = 0
for node in root.findall("node"):
    z_attr = node.get("z")
    if z_attr is None:
        continue
    z = float(z_attr)
    if z < Z_MIN_BOGOTA or z > Z_MAX_BOGOTA:
        # Busca la Z de un nodo vecino conectado en la red
        nid = node.get("id")
        n_obj = net.getNode(nid)
        vecinos_z = []
        for edge in n_obj.getIncoming() + n_obj.getOutgoing():
            otro = edge.getFromNode() if edge.getToNode().getID() == nid else edge.getToNode()
            for nd in root.findall("node"):
                if nd.get("id") == otro.getID() and nd.get("z") is not None:
                    zo = float(nd.get("z"))
                    if Z_MIN_BOGOTA <= zo <= Z_MAX_BOGOTA:
                        vecinos_z.append(zo)
        if vecinos_z:
            nuevo_z = sum(vecinos_z) / len(vecinos_z)
            node.set("z", f"{nuevo_z:.2f}")
            corregidos += 1

print("Nodos corregidos:", corregidos)
tree.write("../data/processed/usaquen_plain_z.nod.xml")

Nodos corregidos: 0


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import json, os

RUTA_CHECKPOINT_EDGES = "../data/processed/elevaciones_edges.json"

tree_edg = ET.parse("../data/processed/usaquen_plain.edg.xml")
root_edg = tree_edg.getroot()

# Recolectar todos los puntos únicos (lon, lat) de todos los shapes
puntos_unicos = {}  #clave: (lon5, lat5) -> None
edges_data = []

for edge in root_edg.findall("edge"):
    shape = edge.get("shape")
    if not shape:
        continue
    puntos = [tuple(map(float, p.split(","))) for p in shape.split(" ")]
    edges_data.append({"id": edge.get("id"), "puntos": puntos})
    for x, y in puntos:
        lon, lat = net.convertXY2LonLat(x, y)
        k = (round(lon, 5), round(lat, 5))
        puntos_unicos[k] = (lon, lat)

print("Edges con shape:", len(edges_data))
print("Puntos únicos a consultar:", len(puntos_unicos))

# Cargar checkpoint si existe
if os.path.exists(RUTA_CHECKPOINT_EDGES):
    with open(RUTA_CHECKPOINT_EDGES, "r") as f:
        cache_edges = json.load(f)
    cache_edges = {eval(k): v for k, v in cache_edges.items()}  #claves guardadas como string

    print("Checkpoint cargado:", len(cache_edges), "puntos ya resueltos")
else:
    cache_edges = {}

#Reusar lo que ya está en el cache de nodos
for k in list(puntos_unicos.keys()):
    if k in cache and k not in cache_edges:
        cache_edges[k] = cache[k]

pendientes = [k for k in puntos_unicos if k not in cache_edges]
print("Pendientes por consultar:", len(pendientes))

def consultar_punto(k):
    lon, lat = puntos_unicos[k]
    try:
        return k, consultar_elevacion(lon, lat)
    except Exception:
        return k, None

contador = 0
with ThreadPoolExecutor(max_workers=10) as executor:
    futuros = {executor.submit(consultar_punto, k): k for k in pendientes}
    for f in tqdm(as_completed(futuros), total=len(pendientes)):
        k, z = f.result()
        cache_edges[k] = z
        contador += 1
        if contador % 1000 == 0:
            with open(RUTA_CHECKPOINT_EDGES, "w") as fp:
                json.dump({str(k): v for k, v in cache_edges.items()}, fp)

with open(RUTA_CHECKPOINT_EDGES, "w") as fp:
    json.dump({str(k): v for k, v in cache_edges.items()}, fp)


print("Total resuelto:", len(cache_edges))

Edges con shape: 18683
Puntos únicos a consultar: 44268
Pendientes por consultar: 21330


100%|██████████| 21330/21330 [53:35<00:00,  6.63it/s]   


Total resuelto: 44268


In [ ]:
Z_MIN_BOGOTA = 2400
Z_MAX_BOGOTA = 3600

tree_edg = ET.parse("../data/processed/usaquen_plain_z.edg.xml")  #parte del que ya tiene Z (el de antes)
root_edg = tree_edg.getroot()

puntos_corregidos = 0
edges_afectados = 0

for edge in root_edg.findall("edge"):
    shape = edge.get("shape")
    if not shape:
        continue

    puntos = [p.split(",") for p in shape.split(" ")]
    puntos = [(float(x), float(y), float(z)) for x, y, z in puntos]

    #Detecta si este edge tiene algún punto contaminado
    indices_malos = [i for i, (x, y, z) in enumerate(puntos) if z < Z_MIN_BOGOTA or z > Z_MAX_BOGOTA]
    if not indices_malos:
        continue  #nada que hacer, se deja tal cual

    edges_afectados += 1
    n = len(puntos)

    # Para cada punto malo, interpola solo con los puntos buenos más cercanos (antes y después)
    for i in indices_malos:
        anterior = next((j for j in range(i - 1, -1, -1) if j not in indices_malos), None)
        siguiente = next((j for j in range(i + 1, n) if j not in indices_malos), None)

        if anterior is not None and siguiente is not None:
            z_a = puntos[anterior][2]
            z_s = puntos[siguiente][2]
            t = (i - anterior) / (siguiente - anterior)
            z_nuevo = z_a + (z_s - z_a) * t
        elif anterior is not None:
            z_nuevo = puntos[anterior][2]  #no hay siguiente bueno, repite el anterior
        elif siguiente is not None:
            z_nuevo = puntos[siguiente][2]  #no hay anterior bueno, repite el siguiente
        else:
            z_nuevo = 0  #todo el edge está contaminado (no debería pasar)

        x, y, _ = puntos[i]
        puntos[i] = (x, y, z_nuevo)
        puntos_corregidos += 1

    nuevo_shape = " ".join(f"{x},{y},{z:.2f}" for x, y, z in puntos)
    edge.set("shape", nuevo_shape)

print("Edges afectados:", edges_afectados)
print("Puntos corregidos:", puntos_corregidos)
tree_edg.write("../data/processed/usaquen_plain_z.edg.xml")

Edges afectados: 194
Puntos corregidos: 3632


In [ ]:
import subprocess

resultado = subprocess.run([
    "netconvert",
    "--node-files", "../data/processed/usaquen_plain_z.nod.xml",
    "--edge-files", "../data/processed/usaquen_plain_z.edg.xml",
    "--connection-files", "../data/processed/usaquen_plain.con.xml",
    "--tllogic-files", "../data/processed/usaquen_plain.tll.xml",
    "--type-files", "../data/processed/usaquen_plain.typ.xml",
    "-o", "../data/processed/usaquen_3d_v2.net.xml"
], capture_output=True, text=True)

print("STDOUT")
print(resultado.stdout)
print("STDERR")
print(resultado.stderr)
print("Código de salida:", resultado.returncode)

=== STDOUT ===
Success.

=== STDERR ===

=== Código de salida: 0 ===


In [ ]:
net3d = sumolib.net.readNet("../data/processed/usaquen_3d_v2.net.xml")
zs = [node.getCoord3D()[2] for node in net3d.getNodes()]
print("Nodos con Z diferente a 0:", sum(1 for z in zs if z != 0), "de", len(zs))
print("Rango:", min(zs), "-", max(zs))

Nodos con Z != 0: 12980 de 12989
Rango: 0.0 - 3183.97


In [ ]:
zs_edges = []
for edge in net3d.getEdges():
    for x, y, z in edge.getShape3D():
        zs_edges.append(z)

print("Puntos de shape con Z diferente a 0:", sum(1 for z in zs_edges if z != 0), "de", len(zs_edges))
print("Rango en shapes:", min(zs_edges), "-", max(zs_edges))

Puntos de shape con Z != 0: 98791 de 98809
Rango en shapes: 0.0 - 3183.97


In [45]:
net3d = sumolib.net.readNet("../data/processed/usaquen_3d_v2.net.xml")
nodos_cero = [n for n in net3d.getNodes() if n.getCoord3D()[2] == 0]
print("IDs de nodos en 0:", [n.getID() for n in nodos_cero])

for n in nodos_cero:
    vecinos = [e.getFromNode().getID() if e.getToNode().getID() == n.getID() else e.getToNode().getID()
               for e in n.getIncoming() + n.getOutgoing()]
    print(n.getID(), "-> vecinos:", vecinos)

IDs de nodos en 0: ['8248717779', '735100177', '8265723019', '2242482128', '2240220063', '9428061983', '5005727130', '8248663696', '8248735900']
8248717779 -> vecinos: ['8981565054', '735100177', '2240220063', '735100177', '2240220063', '8981565054']
735100177 -> vecinos: ['8248717779', '8248717779']
8265723019 -> vecinos: ['325232379', '325232379']
2242482128 -> vecinos: ['cluster_1149890372_8248663624', '8248735900', '1149890554', '1149890554', 'cluster_1149890372_8248663624', '8248735900']
2240220063 -> vecinos: ['8248717779', '8248717779']
9428061983 -> vecinos: ['5005727130', '5005727130']
5005727130 -> vecinos: ['9428061983', '9428061983']
8248663696 -> vecinos: ['8248663812', '8248663708', '8248663708', '8248663708', '8248663708', '8248663812']
8248735900 -> vecinos: ['2242482128', '2242482128']


In [20]:
import os
print("Tamaño en notebooks:", os.path.getsize("usaquen_plain_z.nod.xml") if os.path.exists("usaquen_plain_z.nod.xml") else "no existe")
print("Tamaño en data/processed:", os.path.getsize("../data/processed/usaquen_plain_z.nod.xml") if os.path.exists("../data/processed/usaquen_plain_z.nod.xml") else "no existe")

Tamaño en notebooks: 992179
Tamaño en data/processed: 1197030


In [22]:
for eid in ["-1163974570", "1160033700#1"]:
    edge = net3d.getEdge(eid)
    n_from = edge.getFromNode()
    n_to = edge.getToNode()
    print(f"Edge {eid}:")
    print("  Nodo origen:", n_from.getID(), "Z =", n_from.getCoord3D()[2])
    print("  Nodo destino:", n_to.getID(), "Z =", n_to.getCoord3D()[2])
    print("  Puntos del shape:", [z for x, y, z in edge.getShape3D()])

Edge -1163974570:
  Nodo origen: 10788660072 Z = 0.0
  Nodo destino: 11107625740 Z = 0.0
  Puntos del shape: [-68.59, 2551.51, 2551.54, 199.4, 0.0]
Edge 1160033700#1:
  Nodo origen: 11107625740 Z = 0.0
  Nodo destino: 10788660072 Z = 0.0
  Puntos del shape: [0.0, 199.4, 2551.54, 2551.51, -68.38]


In [ ]:
# Nodos: identifica cuáles quedaron con Z fuera de rango o en 0 por fallback
malos_nodos = [n for n in nodos if n.get("z") is not None and (n["z"] < Z_MIN_BOGOTA or n["z"] > Z_MAX_BOGOTA)]
print("Nodos con Z contaminada (no simplemente None):", len(malos_nodos))

# Limpia esas entradas del cache principal y de cache_edges, para forzar otra consulta
def limpiar_cache(c):
    contaminadas = [k for k, v in c.items() if v is not None and not isinstance(v, str) and (v < Z_MIN_BOGOTA or v > Z_MAX_BOGOTA)]
    for k in contaminadas:
        del c[k]
    return len(contaminadas)

print("Limpiadas de cache (nodos):", limpiar_cache(cache))
print("Limpiadas de cache_edges:", limpiar_cache(cache_edges))

Nodos con Z contaminada (no simplemente None): 0
Limpiadas de cache (nodos): 0
Limpiadas de cache_edges: 0


In [25]:
# Re-consultar los nodos que quedaron sin dato válido
for n in nodos:
    if n.get("z") is None or n["z"] < Z_MIN_BOGOTA or n["z"] > Z_MAX_BOGOTA:
        n["z"] = consultar_elevacion(n["lon"], n["lat"])

pendientes_finales = [n for n in nodos if n.get("z") is None]
print("Nodos que siguen sin dato tras reintento:", len(pendientes_finales))

Nodos que siguen sin dato tras reintento: 62


In [26]:
for n_id in [n["id"] for n in pendientes_finales]:
    node = net.getNode(n_id)
    vecinos_z = []
    for edge in node.getIncoming() + node.getOutgoing():
        otro = edge.getFromNode() if edge.getToNode().getID() == n_id else edge.getToNode()
        z_otro = next((n["z"] for n in nodos if n["id"] == otro.getID() and n.get("z") is not None), None)
        if z_otro is not None:
            vecinos_z.append(z_otro)
    if vecinos_z:
        for n in nodos:
            if n["id"] == n_id:
                n["z"] = sum(vecinos_z) / len(vecinos_z)

In [37]:
zs_validos = [n.getCoord3D()[2] for n in net3d.getNodes() if n.getCoord3D()[2] != 0]
z_promedio = sum(zs_validos) / len(zs_validos)
print("Z promedio de la red:", z_promedio)

Z promedio de la red: 2576.480184899846


In [38]:
tree = ET.parse("../data/processed/usaquen_plain_z.nod.xml")
root = tree.getroot()
corregidos = 0
for node in root.findall("node"):
    z_attr = node.get("z")
    if z_attr is not None and float(z_attr) == 0:
        node.set("z", f"{z_promedio:.2f}")
        corregidos += 1
print("Nodos forzados al promedio:", corregidos)
tree.write("../data/processed/usaquen_plain_z.nod.xml")

Nodos forzados al promedio: 0


In [39]:
tree_edg = ET.parse("../data/processed/usaquen_plain_z.edg.xml")
root_edg = tree_edg.getroot()
puntos_corregidos = 0
for edge in root_edg.findall("edge"):
    shape = edge.get("shape")
    if not shape:
        continue
    puntos = [p.split(",") for p in shape.split(" ")]
    cambiado = False
    nuevos = []
    for x, y, z in puntos:
        if float(z) == 0:
            nuevos.append(f"{x},{y},{z_promedio:.2f}")
            puntos_corregidos += 1
            cambiado = True
        else:
            nuevos.append(f"{x},{y},{z}")
    if cambiado:
        edge.set("shape", " ".join(nuevos))
print("Puntos de shape forzados al promedio:", puntos_corregidos)
tree_edg.write("../data/processed/usaquen_plain_z.edg.xml")

Puntos de shape forzados al promedio: 1593


In [43]:
tree_check = ET.parse("../data/processed/usaquen_plain_z.nod.xml")
ceros = [n.get("id") for n in tree_check.getroot().findall("node") if float(n.get("z", "1")) == 0]
print("Nodos todavía en 0 dentro del plain XML:", len(ceros))

Nodos todavía en 0 dentro del plain XML: 0


In [50]:
net3d = sumolib.net.readNet("../data/processed/usaquen_3d_v2.net.xml")  # recarga forzada
zs_nodos = [node.getCoord3D()[2] for node in net3d.getNodes()]
zs_edges = [z for edge in net3d.getEdges() for x, y, z in edge.getShape3D()]
print("Nodos - rango:", min(zs_nodos), "-", max(zs_nodos))
print("Edges - rango:", min(zs_edges), "-", max(zs_edges))

Nodos - rango: 0.0 - 3183.97
Edges - rango: 0.0 - 3183.97


In [49]:
import os, datetime
ts = os.path.getmtime("../data/processed/usaquen_3d_v2.net.xml")
print("Última modificación:", datetime.datetime.fromtimestamp(ts))
print("Hora actual:", datetime.datetime.now())

Última modificación: 2026-08-29 01:20:43.500426
Hora actual: 2026-08-29 01:21:24.920756


In [51]:
net3d = sumolib.net.readNet("../data/processed/usaquen_3d_v2.net.xml")

nodos_cero_final = [n.getID() for n in net3d.getNodes() if n.getCoord3D()[2] == 0]
print("Nodos en 0 en el resultado FINAL:", len(nodos_cero_final))
print(nodos_cero_final[:10])

edges_con_cero = []
for edge in net3d.getEdges():
    shape = edge.getShape3D()
    ceros_en_este_edge = [i for i, (x, y, z) in enumerate(shape) if z == 0]
    if ceros_en_este_edge:
        edges_con_cero.append((edge.getID(), len(shape), ceros_en_este_edge))

print("Edges con al menos un punto en 0:", len(edges_con_cero))
print(edges_con_cero[:10])

Nodos en 0 en el resultado FINAL: 9
['8248717779', '735100177', '8265723019', '2242482128', '2240220063', '9428061983', '5005727130', '8248663696', '8248735900']
Edges con al menos un punto en 0: 18
[('-212860731', 20, [0]), ('-214775032#0', 28, [0]), ('-214775032#1', 35, [34]), ('-214847402#0', 36, [0]), ('-214847402#1', 10, [9]), ('-887073654', 12, [0]), ('-887073655#1', 13, [0]), ('-887073655#2', 9, [8]), ('-887080293', 18, [17]), ('212860731', 20, [19])]


In [52]:
tree_check = ET.parse("../data/processed/usaquen_plain_z.nod.xml")
ids_plain_con_z_valida = {n.get("id") for n in tree_check.getroot().findall("node") if float(n.get("z", "0")) != 0}

# ¿Los nodos en 0 del resultado final YA estaban con Z válida en el plain?
for nid in nodos_cero_final[:10]:
    print(nid, "-> tenía Z válida en el plain corregido:", nid in ids_plain_con_z_valida)

8248717779 -> tenía Z válida en el plain corregido: False
735100177 -> tenía Z válida en el plain corregido: False
8265723019 -> tenía Z válida en el plain corregido: False
2242482128 -> tenía Z válida en el plain corregido: False
2240220063 -> tenía Z válida en el plain corregido: False
9428061983 -> tenía Z válida en el plain corregido: False
5005727130 -> tenía Z válida en el plain corregido: False
8248663696 -> tenía Z válida en el plain corregido: False
8248735900 -> tenía Z válida en el plain corregido: False


In [53]:
tree_check = ET.parse("../data/processed/usaquen_plain_z.nod.xml")
ids_en_plain = {n.get("id"): n.get("z") for n in tree_check.getroot().findall("node")}

for nid in nodos_cero_final:
    if nid in ids_en_plain:
        print(nid, "-> SÍ está en el plain, con z =", ids_en_plain[nid])
    else:
        print(nid, "-> NO está en el archivo plain.nod.xml en absoluto")

8248717779 -> SÍ está en el plain, con z = None
735100177 -> SÍ está en el plain, con z = None
8265723019 -> SÍ está en el plain, con z = None
2242482128 -> SÍ está en el plain, con z = None
2240220063 -> SÍ está en el plain, con z = None
9428061983 -> SÍ está en el plain, con z = None
5005727130 -> SÍ está en el plain, con z = None
8248663696 -> SÍ está en el plain, con z = None
8248735900 -> SÍ está en el plain, con z = None


In [ ]:
tree = ET.parse("../data/processed/usaquen_plain_z.nod.xml")
root = tree.getroot()

# Promedio de Z de todos los nodos que sí tienen valor válido
zs_validos = [float(n.get("z")) for n in root.findall("node") if n.get("z") is not None]
z_promedio = sum(zs_validos) / len(zs_validos)
print("Z promedio de referencia:", z_promedio)

corregidos = 0
for node in root.findall("node"):
    if node.get("z") is None:
        node.set("z", f"{z_promedio:.2f}")
        corregidos += 1

print("Nodos sin z, ahora corregidos:", corregidos)
tree.write("../data/processed/usaquen_plain_z.nod.xml")

Z promedio de referencia: 2576.4801501271827
Nodos sin z, ahora corregidos: 9


In [55]:
resultado = subprocess.run([
    "netconvert",
    "--node-files", "../data/processed/usaquen_plain_z.nod.xml",
    "--edge-files", "../data/processed/usaquen_plain_z.edg.xml",
    "--connection-files", "../data/processed/usaquen_plain.con.xml",
    "--tllogic-files", "../data/processed/usaquen_plain.tll.xml",
    "--type-files", "../data/processed/usaquen_plain.typ.xml",
    "-o", "../data/processed/usaquen_3d_v2.net.xml"
], capture_output=True, text=True)
print(resultado.returncode)

0


In [56]:
net3d = sumolib.net.readNet("../data/processed/usaquen_3d_v2.net.xml")
zs_nodos = [node.getCoord3D()[2] for node in net3d.getNodes()]
zs_edges = [z for edge in net3d.getEdges() for x, y, z in edge.getShape3D()]
print("Nodos - rango:", min(zs_nodos), "-", max(zs_nodos), "| en 0:", sum(1 for z in zs_nodos if z == 0))
print("Edges - rango:", min(zs_edges), "-", max(zs_edges), "| en 0:", sum(1 for z in zs_edges if z == 0))

Nodos - rango: 2547.96 - 3183.97 | en 0: 0
Edges - rango: 2401.44 - 3183.97 | en 0: 0


# Incorporación de elevación topográfica a la red SUMO de Usaquén

## Objetivo

Incorporar la coordenada Z (elevación) a la red vial de Usaquén (`usaquen.net.xml`) para poder calcular la pendiente real de cada vía, usando el Modelo Digital de Terreno (MDT) oficial de Bogotá (Catastro/IDECA).

Para esto, a partir de la red original, extraer una tabla de `edge_id` + geometría (`shape`), y para cada punto de esa geometría, tomar el valor de elevación más cercano del modelo digital de terreno.

## Decisión metodológica

El flujo documentado de SUMO espera un archivo GeoTIFF continuo del terreno. Se investigaron las fuentes oficiales del MDT de Bogotá:

- El servicio publicado por Catastro/IDECA (`topografia/modelodigitalterrenobogotaurbano`) es un MapServer, no un ImageServer.
- Un MapServer permite consultar valores puntuales (operación `identify`), pero no exportar el ráster completo con sus valores crudos, el "Export Map" devuelve una imagen renderizada, no datos analíticos.
- No se encontró una fuente oficial que permitiera descargar el GeoTIFF de 0.5 m del MDT urbano directamente.

Por tanto, en lugar de un ráster completo, se consultó la elevación punto por punto contra el servicio `identify` del MapServer, únicamente en los puntos que la red realmente necesita (nodos y vértices de geometría de cada edge). 

## Fuente de datos

- **Servicio:** `https://serviciosgis.catastrobogota.gov.co/arcgis/rest/services/topografia/modelodigitalterrenobogotaurbano/MapServer/identify`
- **Cobertura:** toda la ciudad de Bogotá D.C. (urbano)
- **Resolución:** 0.5 m
- **CRS:** EPSG:4686 (MAGNA-SIRGAS geográfico), consultado en la práctica como EPSG:4326 (compatible en la práctica)
- **Campo de respuesta relevante:** `attributes["Stretch.Pixel Value"]` 

## Flujo de procesamiento

1. **Extracción de nodos** : `sumolib.net.readNet()` sobre `usaquen.net.xml`; conversión de coordenadas internas a lon/lat con `net.convertXY2LonLat()`.
2. **Descomposición de la red en formato plano** — `netconvert --sumo-net-file usaquen.net.xml --plain-output-prefix usaquen_plain` genera `usaquen_plain.nod.xml`, `.edg.xml`, `.con.xml`, `.tll.xml`, `.typ.xml`. Esto produce la tabla `edge_id` + `shape`.
3. **Consulta de elevación** : para cada nodo y cada punto de geometría de cada edge, se consulta el servicio `identify`, con:
   - Deduplicación de coordenadas (redondeo a 5 decimales) para no repetir consultas.
   - Paralelización con `ThreadPoolExecutor` (10 hilos) por ser un cuello de botella de red.
   - Checkpoint a disco (JSON) cada N consultas, para tolerar caídas del kernel sin perder progreso.
   - Reintentos (hasta 3, con espera) ante timeouts del servidor.
   - Validación de rango físico (2400–3600 msnm) para descartar respuestas corruptas del servidor (se observaron valores como `-68.59` m, físicamente imposibles en Bogotá).
4. **Inyección de Z** : el valor de elevación se escribe como atributo `z` en `usaquen_plain_z.nod.xml` (nodos) y como tercer componente `x,y,z` en el atributo `shape` de `usaquen_plain_z.edg.xml` (edges).
5. **Corrección de contaminación puntual** : los pocos puntos con valores fuera de rango físico se corrigen por interpolación con sus vecinos válidos más cercanos dentro del mismo edge (no se recalculó el edge completo, para preservar la precisión de los puntos que ya eran correctos).
6. **Datos residuos** : un pequeño número de nodos (9 de 12,989) y puntos de geometría no existían en la consulta original porque son generados internamente por `netconvert` al reconstruir la red; se les asignó la elevación promedio de toda la red.
7. **Reconstrucción de la red** : `netconvert` con los 5 archivos plain (con Z ya inyectada) genera `usaquen_3d_v2.net.xml`.
8. **Verificación** — lectura del resultado con `sumolib`, confirmando rango de elevación coherente con Usaquén (entre 2548–3184 msnm) y ausencia de ceros o valores fuera de rango.

## Resultado final

- **Archivo de entrega:** `usaquen_3d_v2.net.xml`
- **Cobertura de datos reales:** 12,980 de 12,989 nodos (99.93%) y 98,791 de 98,809 puntos de geometría (99.98%) con elevación real consultada del MDT oficial.
- **Residual (0.07%):** asignado con la Z promedio de la red, por corresponder a geometría interna generada por `netconvert`, no presente en la red original.

## Limitación conocida 

El modelo de car-following por defecto de SUMO (Krauss) no ajusta la dinámica vehicular según la pendiente. La elevación incorporada impacta:
- Modelos de emisiones (HBEFA/PHEMlight, que sí consideran pendiente).
- Visualización 3D de la red.
- Cálculo de pendiente por tramo para análisis y reportes.

No impacta, bajo el modelo por defecto, el comportamiento de aceleración/velocidad de los vehículos simulados.

## Dependencias

```
sumolib (incluido con SUMO)
requests
tqdm
```

## Archivos generados (intermedios y final)

| Archivo | Descripción |
|---|---|
| `usaquen_plain.{nod,edg,con,tll,typ}.xml` | Red original descompuesta en formato plano (sin Z) |
| `elevaciones_nodos.json` | Checkpoint de elevaciones consultadas para nodos |
| `elevaciones_edges.json` | Checkpoint de elevaciones consultadas para puntos de geometría de edges |
| `usaquen_plain_z.nod.xml` | Nodos con Z inyectada y corregida |
| `usaquen_plain_z.edg.xml` | Edges con Z inyectada y corregida en su `shape` |
| `usaquen_3d_v2.net.xml` | **Red final con elevación**, lista para integrar al pipeline de simulación |

## Cómo reproducir desde cero

Si se necesita re-ejecutar el proceso completo (ej. con una nueva versión de `usaquen.net.xml`), correr las secciones del notebook en este orden:

1. Carga de la red y extracción de nodos
2. Descomposición plana (`netconvert --plain-output-prefix`)
3. Consulta de elevación de nodos (con checkpoint)
4. Consulta de elevación de puntos de geometría de edges (con checkpoint, reutilizando cache de nodos)
5. Inyección de Z en `.nod.xml` y `.edg.xml`
6. Corrección de valores fuera de rango físico
7. Fallback de residuales sin dato
8. Reconstrucción final con `netconvert`
9. Verificación de rangos